# SoundStream Codec Demo

This notebook downloads a pre-trained SoundStream model and applies it to any audio URL.

Just provide an audio URL in the cell below and run all cells (Runtime → Run all).

## 1. Clone repository and install dependencies

In [ ]:
!git clone https://github.com/AlexanderPlotnikovv/SoundStreamHW.git
%cd SoundStreamHW
!pip install -q -r requirements.txt 2>&1 | tail -5

## 2. Download pre-trained checkpoint from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="AlexPlotnikovTech/soundstream-libri",
    repo_type="model",
    local_dir="soundstream_libri",
)

print("Checkpoint downloaded.")

## 3. Provide your audio URL

Replace the URL below with your own audio file URL. The codec works on speech sampled at any rate (will be resampled to 16 kHz).

In [ ]:
audio_url = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

print(f"Downloading: {audio_url}")
!wget -q -O input_audio "$audio_url"

## 4. Run the codec

Load the model, pass the audio through the encoder→RVQ→decoder pipeline, and save the reconstruction.

In [ ]:
import torch
import torchaudio
import torch.nn.functional as F
from src.model.soundstream import SoundStream

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SoundStream()
ckpt = torch.load("soundstream_libri/model_best.pth", map_location=device)
state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
model.load_state_dict(state_dict)
model = model.to(device).eval()

wav, sr = torchaudio.load("input_audio")
if sr != 16000:
    wav = torchaudio.transforms.Resample(sr, 16000)(wav)
if wav.shape[0] > 1:
    wav = wav.mean(dim=0, keepdim=True)

L = wav.shape[-1]
pad = (200 - L % 200) % 200
if pad > 0:
    wav = F.pad(wav, (0, pad), mode="replicate")

x = wav.unsqueeze(0).to(device)
with torch.no_grad():
    x_hat, _, _, _ = model(x)
x_hat = x_hat[0].cpu()[:, :L]

torchaudio.save("output.wav", x_hat, 16000)
print(f"Reconstruction saved. Audio length: {L / 16000:.2f}s")

## 5. Listen and compare

In [ ]:
from IPython.display import Audio, display

print("=== Original audio ===")
display(Audio("input_audio"))

print("=== Reconstructed audio (via SoundStream codec) ===")
display(Audio("output.wav"))